In [29]:
import warnings
from statsbombpy.api_client import NoAuthWarning
from socceraction.data.statsbomb import StatsBombLoader
import pandas as pd
from statsbombpy import sb
import numpy as np
import json
import socceraction.spadl as spadl

warnings.filterwarnings("ignore", category=NoAuthWarning)
warnings.filterwarnings("ignore", message="credentials were not supplied")

pd.set_option("display.max_columns", None)  # Show all columns

In [30]:
df = pd.read_csv("data/stage.csv", index_col = 0)

In [31]:
df

,game_id,original_event_id,index,period_id,minute,second,duration,start_x,start_y,type_name,result_name,is_home,home_yellow_count,away_yellow_count,home_red_count,away_red_count,home_goal_count,away_goal_count
0,3754217,5f9bdb6d-0379-4f42-9f53-29777f166db5,5,1,0,0,0.386674,51.66875,34.0425,pass,success,False,0,0,0,0,0,0
1,3754217,e5d7db8c-d134-4392-a109-9f6efc7f7225,7,1,0,1,1.499254,53.15625,35.7425,pass,success,False,0,0,0,0,0,0
2,3754217,7ec3ada2-d200-4f0b-ab86-f5a464dc4aa7,9,1,0,2,2.672921,66.80625,32.4275,dribble,success,False,0,0,0,0,0,0
3,3754217,fd5039d0-8dd4-417c-9eac-decd2d267e93,10,1,0,5,0.986454,61.73125,35.4025,pass,success,False,0,0,0,0,0,0
4,3754217,d8a6875b-79c2-4f3b-bb08-38cda23763ec,12,1,0,6,0.851193,49.83125,41.5225,dribble,success,False,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
740505,3754078,8cf002ff-68ff-44a5-8882-56d2bd753075,3342,2,94,8,1.424035,50.26875,7.6925,dribble,success,False,1,3,1,0,2,2
740506,3754078,88aafe86-2436-4bc4-b396-49bb0c928656,3343,2,94,9,1.400675,49.39375,8.5425,pass,success,False,1,3,1,0,2,2
740507,3754078,05a25779-e472-4f3c-a838-1b95397bc7fb,3345,2,94,10,0.569196,40.99375,5.5675,dribble,success,False,1,3,1,0,2,2
740508,3754078,82b06de4-7851-4c07-b475-d658f6dd3f37,3347,2,94,11,0.000000,38.98125,5.5675,foul,fail,True,1,3,1,0,2,2


In [32]:
df[["start_x", "start_y", 'duration']] = df[["start_x", "start_y", 'duration']].round(0).astype(int)

In [33]:
df['is_home'] = df['is_home'].astype(int)

In [34]:
pieces = []
seen_games = set()

for (game_id, period_id), half in df.groupby(["game_id", "period_id"], sort=False):
    # Add kick_off sentinel at the start of each game's first period
    if game_id not in seen_games and period_id == 1:
        seen_games.add(game_id)
        
        kick_off_row = pd.DataFrame([[-2] * len(df.columns)], columns=df.columns)
        kick_off_row["game_id"] = game_id
        kick_off_row["period_id"] = period_id
        kick_off_row["index"] = 0  # Will be recalculated later
        kick_off_row["minute"] = 0
        kick_off_row["second"] = 0
        kick_off_row["type_name"] = 'kick_off'
        pieces.append(kick_off_row)
    
    # Append the actual half data
    pieces.append(half)
    
    # Add end_row sentinel
    last = half.iloc[-1]
    end_row = pd.DataFrame([[-2] * len(df.columns)], columns=df.columns)
    end_row["game_id"] = game_id
    end_row["period_id"] = period_id
    end_row["index"] = last["index"] 
    end_row["minute"] = last["minute"]
    end_row["second"] = last["second"]
    end_row["type_name"] = 'half_end' if period_id == 1 else 'full_end'
    pieces.append(end_row)

b_df = pd.concat(pieces, ignore_index=True)
b_df["index"] = b_df.groupby("game_id", sort=False).cumcount()

In [35]:
# Compute time_elapsed (seconds since previous row within the same game/period),
# alongside duration (kept, not replaced). Must run after sentinel rows (including kick_off) are in b_df,
# so period 1's first real event is diffed against kick_off's total_seconds == 0,
# and period 2's first real event (no prior row in its group) is filled with 0.
b_df['total_seconds'] = b_df['minute'] * 60 + b_df['second']
b_df['time_elapsed'] = b_df.groupby(['game_id', 'period_id'])['total_seconds'].diff().fillna(0)
b_df['time_elapsed'] = b_df['time_elapsed'].round(0).astype(int)
b_df = b_df.drop(columns=['total_seconds'])

In [36]:
b_df['time_elapsed'] = b_df['time_elapsed'].clip(upper=105)

In [37]:
b_df['type_name'].unique()

array(['kick_off', 'pass', 'dribble', 'take_on', 'foul', 'freekick_short',
       'bad_touch', 'tackle', 'corner_crossed', 'shot', 'clearance',
       'cross', 'throw_in', 'goalkick', 'keeper_save', 'freekick_crossed',
       'interception', 'keeper_claim', 'half_end', 'shot_freekick',
       'corner_short', 'full_end', 'shot_penalty', 'keeper_punch'],
      dtype=object)

In [38]:
b_df

,game_id,original_event_id,index,period_id,minute,second,duration,start_x,start_y,type_name,result_name,is_home,home_yellow_count,away_yellow_count,home_red_count,away_red_count,home_goal_count,away_goal_count,time_elapsed
0,3754217,-2,0,1,0,0,-2,-2,-2,kick_off,-2,-2,-2,-2,-2,-2,-2,-2,0
1,3754217,5f9bdb6d-0379-4f42-9f53-29777f166db5,1,1,0,0,0,52,34,pass,success,0,0,0,0,0,0,0,0
2,3754217,e5d7db8c-d134-4392-a109-9f6efc7f7225,2,1,0,1,1,53,36,pass,success,0,0,0,0,0,0,0,1
3,3754217,7ec3ada2-d200-4f0b-ab86-f5a464dc4aa7,3,1,0,2,3,67,32,dribble,success,0,0,0,0,0,0,0,1
4,3754217,fd5039d0-8dd4-417c-9eac-decd2d267e93,4,1,0,5,1,62,35,pass,success,0,0,0,0,0,0,0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
741645,3754078,88aafe86-2436-4bc4-b396-49bb0c928656,1897,2,94,9,1,49,9,pass,success,0,1,3,1,0,2,2,1
741646,3754078,05a25779-e472-4f3c-a838-1b95397bc7fb,1898,2,94,10,1,41,6,dribble,success,0,1,3,1,0,2,2,1
741647,3754078,82b06de4-7851-4c07-b475-d658f6dd3f37,1899,2,94,11,0,39,6,foul,fail,1,1,3,1,0,2,2,1
741648,3754078,58ecdddd-bbcb-41a1-a5fd-132b6d53df28,1900,2,94,26,1,37,5,freekick_short,success,0,1,3,1,0,2,2,15


In [39]:
event_types = b_df['type_name'].unique()
e_lookup = {}
e_reverse_lookup = {}

for i, event in enumerate(event_types):
    e_lookup[event] = i
    e_reverse_lookup[i] = event

b_df['event'] = b_df['type_name'].map(e_lookup)

result_types = [r for r in b_df["result_name"].unique() if r != -2]
r_lookup = {}
r_reverse_lookup = {}

for i, result in enumerate(result_types):
    r_lookup[result] = i
    r_reverse_lookup[i] = result

b_df["result"] = b_df["result_name"].map(r_lookup)
b_df.loc[b_df["result_name"] == -2, "result"] = -2
b_df["result"] = b_df["result"].astype(int)

In [40]:
with open('data/e_lookup.json', 'w') as f:
    json.dump(e_lookup, f)

with open('data/r_lookup.json', 'w') as f:
    json.dump(r_lookup, f)

In [41]:
b_df

,game_id,original_event_id,index,period_id,minute,second,duration,start_x,start_y,type_name,result_name,is_home,home_yellow_count,away_yellow_count,home_red_count,away_red_count,home_goal_count,away_goal_count,time_elapsed,event,result
0,3754217,-2,0,1,0,0,-2,-2,-2,kick_off,-2,-2,-2,-2,-2,-2,-2,-2,0,0,-2
1,3754217,5f9bdb6d-0379-4f42-9f53-29777f166db5,1,1,0,0,0,52,34,pass,success,0,0,0,0,0,0,0,0,1,0
2,3754217,e5d7db8c-d134-4392-a109-9f6efc7f7225,2,1,0,1,1,53,36,pass,success,0,0,0,0,0,0,0,1,1,0
3,3754217,7ec3ada2-d200-4f0b-ab86-f5a464dc4aa7,3,1,0,2,3,67,32,dribble,success,0,0,0,0,0,0,0,1,2,0
4,3754217,fd5039d0-8dd4-417c-9eac-decd2d267e93,4,1,0,5,1,62,35,pass,success,0,0,0,0,0,0,0,3,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
741645,3754078,88aafe86-2436-4bc4-b396-49bb0c928656,1897,2,94,9,1,49,9,pass,success,0,1,3,1,0,2,2,1,1,0
741646,3754078,05a25779-e472-4f3c-a838-1b95397bc7fb,1898,2,94,10,1,41,6,dribble,success,0,1,3,1,0,2,2,1,2,0
741647,3754078,82b06de4-7851-4c07-b475-d658f6dd3f37,1899,2,94,11,0,39,6,foul,fail,1,1,3,1,0,2,2,1,4,1
741648,3754078,58ecdddd-bbcb-41a1-a5fd-132b6d53df28,1900,2,94,26,1,37,5,freekick_short,success,0,1,3,1,0,2,2,15,5,0


In [42]:
b_df.describe()

,game_id,index,period_id,minute,second,duration,start_x,start_y,is_home,home_yellow_count,away_yellow_count,home_red_count,away_red_count,home_goal_count,away_goal_count,time_elapsed,event,result
count,7.416500e+05,741650.000000,741650.000000,741650.000000,741650.000000,741650.000000,741650.000000,741650.000000,741650.000000,741650.000000,741650.000000,741650.000000,741650.000000,741650.000000,741650.000000,741650.000000,741650.000000,741650.000000
mean,3.754162e+06,983.210354,1.491929,44.921796,29.292185,1.475672,53.578851,34.033543,0.511549,0.504768,0.611574,0.021106,0.022941,0.676169,0.531419,2.939442,3.009571,0.175687
std,1.098089e+02,576.874498,0.499935,27.044374,17.362852,1.432536,25.030190,20.192913,0.509009,0.828911,0.903947,0.183116,0.180801,0.976391,0.841457,6.762790,3.682224,0.409363
min,3.753972e+06,0.000000,1.000000,0.000000,0.000000,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000,-2.000000,0.000000,0.000000,-2.000000
25%,3.754066e+06,487.000000,1.000000,21.000000,14.000000,1.000000,36.000000,17.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000
50%,3.754161e+06,975.000000,1.000000,45.000000,29.000000,1.000000,54.000000,34.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2.000000,0.000000
75%,3.754257e+06,1463.000000,2.000000,68.000000,44.000000,2.000000,72.000000,52.000000,1.000000,1.000000,1.000000,0.000000,0.000000,1.000000,1.000000,2.000000,2.000000,0.000000
max,3.754351e+06,2399.000000,2.000000,101.000000,59.000000,68.000000,105.000000,68.000000,1.000000,6.000000,9.000000,2.000000,2.000000,6.000000,6.000000,105.000000,23.000000,5.000000


In [43]:
def last_n_events(n, df):
    event_cols = ["is_home", "event", "start_x", "start_y", "time_elapsed", "duration", "result"]
    c = 0

    for c in range(n):
        c += 1
        event_cols_n = [col + '-' + str(c) for col in event_cols]

        df[event_cols_n] = (
            df.sort_values("index").groupby("game_id")[event_cols].shift(c, fill_value=-2)
        )
        
    return df

In [44]:
event_cols = ['is_home', 'event', 'start_x', 'start_y', 'time_elapsed', 'duration', 'result']
event_cols_1 = [col + '1' for col in event_cols]

In [45]:
b_df.sort_values("index").groupby("game_id")[event_cols].shift(1, fill_value=-2)

,is_home,event,start_x,start_y,time_elapsed,duration,result
0,-2,-2,-2,-2,-2,-2,-2
72273,-2,-2,-2,-2,-2,-2,-2
571415,-2,-2,-2,-2,-2,-2,-2
569691,-2,-2,-2,-2,-2,-2,-2
567774,-2,-2,-2,-2,-2,-2,-2
...,...,...,...,...,...,...,...
621106,0,1,57,3,3,1,0
621107,0,2,62,13,0,0,0
621108,0,1,61,13,1,1,0
621109,0,2,63,4,1,2,0


In [46]:
df_3 = last_n_events(3, b_df)

In [47]:
df_3

,game_id,original_event_id,index,period_id,minute,second,duration,start_x,start_y,type_name,result_name,is_home,home_yellow_count,away_yellow_count,home_red_count,away_red_count,home_goal_count,away_goal_count,time_elapsed,event,result,is_home-1,event-1,start_x-1,start_y-1,time_elapsed-1,duration-1,result-1,is_home-2,event-2,start_x-2,start_y-2,time_elapsed-2,duration-2,result-2,is_home-3,event-3,start_x-3,start_y-3,time_elapsed-3,duration-3,result-3
0,3754217,-2,0,1,0,0,-2,-2,-2,kick_off,-2,-2,-2,-2,-2,-2,-2,-2,0,0,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2
1,3754217,5f9bdb6d-0379-4f42-9f53-29777f166db5,1,1,0,0,0,52,34,pass,success,0,0,0,0,0,0,0,0,1,0,-2,0,-2,-2,0,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2
2,3754217,e5d7db8c-d134-4392-a109-9f6efc7f7225,2,1,0,1,1,53,36,pass,success,0,0,0,0,0,0,0,1,1,0,0,1,52,34,0,0,0,-2,0,-2,-2,0,-2,-2,-2,-2,-2,-2,-2,-2,-2
3,3754217,7ec3ada2-d200-4f0b-ab86-f5a464dc4aa7,3,1,0,2,3,67,32,dribble,success,0,0,0,0,0,0,0,1,2,0,0,1,53,36,1,1,0,0,1,52,34,0,0,0,-2,0,-2,-2,0,-2,-2
4,3754217,fd5039d0-8dd4-417c-9eac-decd2d267e93,4,1,0,5,1,62,35,pass,success,0,0,0,0,0,0,0,3,1,0,0,2,67,32,1,3,0,0,1,53,36,1,1,0,0,1,52,34,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
741645,3754078,88aafe86-2436-4bc4-b396-49bb0c928656,1897,2,94,9,1,49,9,pass,success,0,1,3,1,0,2,2,1,1,0,0,2,50,8,4,1,0,1,1,48,11,0,0,1,1,2,47,12,1,0,0
741646,3754078,05a25779-e472-4f3c-a838-1b95397bc7fb,1898,2,94,10,1,41,6,dribble,success,0,1,3,1,0,2,2,1,2,0,0,1,49,9,1,1,0,0,2,50,8,4,1,0,1,1,48,11,0,0,1
741647,3754078,82b06de4-7851-4c07-b475-d658f6dd3f37,1899,2,94,11,0,39,6,foul,fail,1,1,3,1,0,2,2,1,4,1,0,2,41,6,1,1,0,0,1,49,9,1,1,0,0,2,50,8,4,1,0
741648,3754078,58ecdddd-bbcb-41a1-a5fd-132b6d53df28,1900,2,94,26,1,37,5,freekick_short,success,0,1,3,1,0,2,2,15,5,0,1,4,39,6,1,0,1,0,2,41,6,1,1,0,0,1,49,9,1,1,0


In [48]:
df_3.iloc[201031:201033]

,game_id,original_event_id,index,period_id,minute,second,duration,start_x,start_y,type_name,result_name,is_home,home_yellow_count,away_yellow_count,home_red_count,away_red_count,home_goal_count,away_goal_count,time_elapsed,event,result,is_home-1,event-1,start_x-1,start_y-1,time_elapsed-1,duration-1,result-1,is_home-2,event-2,start_x-2,start_y-2,time_elapsed-2,duration-2,result-2,is_home-3,event-3,start_x-3,start_y-3,time_elapsed-3,duration-3,result-3
201031,3754178,dd23626c-597e-4c66-938b-27a236fa9c8e,1183,2,59,27,1,94,21,pass,fail,1,0,0,0,0,1,1,2,1,1,1,2,93,21,2,2,0,0,23,97,27,1,0,1,1,11,96,3,1,1,1
201032,3754178,7309ff3d-8d19-4cb9-ae72-606bc4a92d56,1184,2,67,32,1,64,24,dribble,success,1,0,0,0,0,1,1,105,2,0,1,1,94,21,2,1,1,1,2,93,21,2,2,0,0,23,97,27,1,0,1


In [49]:
def mask_events(df, event_cols):
    n = len(event_cols)
    MASK = -1

    masked_df = df.loc[df.index.repeat(n)].reset_index(drop=True)
    masked_df['step'] = np.tile(np.arange(n), len(df))

    true_values = df[event_cols].loc[df.index.repeat(n)].reset_index(drop=True)
    masked_df['target'] = true_values.to_numpy()[np.arange(len(masked_df)), masked_df['step'].to_numpy()]

    for j, col in enumerate(event_cols):
        masked_df.loc[masked_df['step'] <= j, col] = MASK

    return masked_df

In [50]:
masked_df = mask_events(df_3, event_cols)

In [51]:
masked_df.drop(['game_id', 'original_event_id', 'index', 'type_name', 'result_name', 'step'], axis=1)

,period_id,minute,second,duration,start_x,start_y,is_home,home_yellow_count,away_yellow_count,home_red_count,away_red_count,home_goal_count,away_goal_count,time_elapsed,event,result,is_home-1,event-1,start_x-1,start_y-1,time_elapsed-1,duration-1,result-1,is_home-2,event-2,start_x-2,start_y-2,time_elapsed-2,duration-2,result-2,is_home-3,event-3,start_x-3,start_y-3,time_elapsed-3,duration-3,result-3,target
0,1,0,0,-1,-1,-1,-1,-2,-2,-2,-2,-2,-2,-1,-1,-1,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2
1,1,0,0,-1,-1,-1,-2,-2,-2,-2,-2,-2,-2,-1,-1,-1,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,0
2,1,0,0,-1,-1,-1,-2,-2,-2,-2,-2,-2,-2,-1,0,-1,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2
3,1,0,0,-1,-2,-1,-2,-2,-2,-2,-2,-2,-2,-1,0,-1,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2
4,1,0,0,-1,-2,-2,-2,-2,-2,-2,-2,-2,-2,-1,0,-1,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5191545,2,94,26,-1,-1,-1,-2,-2,-2,-2,-2,-2,-2,-1,21,-1,0,5,37,5,15,1,0,1,4,39,6,1,0,1,0,2,41,6,1,1,0,-2
5191546,2,94,26,-1,-2,-1,-2,-2,-2,-2,-2,-2,-2,-1,21,-1,0,5,37,5,15,1,0,1,4,39,6,1,0,1,0,2,41,6,1,1,0,-2
5191547,2,94,26,-1,-2,-2,-2,-2,-2,-2,-2,-2,-2,-1,21,-1,0,5,37,5,15,1,0,1,4,39,6,1,0,1,0,2,41,6,1,1,0,0
5191548,2,94,26,-1,-2,-2,-2,-2,-2,-2,-2,-2,-2,0,21,-1,0,5,37,5,15,1,0,1,4,39,6,1,0,1,0,2,41,6,1,1,0,-2


In [52]:
masked_df = masked_df[[
    'game_id', 'original_event_id', 'index',
    'is_home', 'event', 'type_name',
    'start_x', 'start_y',
    'time_elapsed', 'duration',
    'result', 'result_name',
    'period_id', 'minute', 'second',
    'home_goal_count', 'away_goal_count',
    'home_red_count', 'away_red_count',
    'home_yellow_count', 'away_yellow_count',
    'is_home-1', 'event-1', 'start_x-1', 'start_y-1', 'time_elapsed-1', 'duration-1', 'result-1',
    'is_home-2', 'event-2', 'start_x-2', 'start_y-2', 'time_elapsed-2', 'duration-2', 'result-2',
    'is_home-3', 'event-3', 'start_x-3', 'start_y-3', 'time_elapsed-3', 'duration-3', 'result-3',
    'step', 'target'
]]

In [53]:
masked_df

,game_id,original_event_id,index,is_home,event,type_name,start_x,start_y,time_elapsed,duration,result,result_name,period_id,minute,second,home_goal_count,away_goal_count,home_red_count,away_red_count,home_yellow_count,away_yellow_count,is_home-1,event-1,start_x-1,start_y-1,time_elapsed-1,duration-1,result-1,is_home-2,event-2,start_x-2,start_y-2,time_elapsed-2,duration-2,result-2,is_home-3,event-3,start_x-3,start_y-3,time_elapsed-3,duration-3,result-3,step,target
0,3754217,-2,0,-1,-1,kick_off,-1,-1,-1,-1,-1,-2,1,0,0,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,0,-2
1,3754217,-2,0,-2,-1,kick_off,-1,-1,-1,-1,-1,-2,1,0,0,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,1,0
2,3754217,-2,0,-2,0,kick_off,-1,-1,-1,-1,-1,-2,1,0,0,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,2,-2
3,3754217,-2,0,-2,0,kick_off,-2,-1,-1,-1,-1,-2,1,0,0,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,3,-2
4,3754217,-2,0,-2,0,kick_off,-2,-2,-1,-1,-1,-2,1,0,0,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,-2,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5191545,3754078,-2,1901,-2,21,full_end,-1,-1,-1,-1,-1,-2,2,94,26,-2,-2,-2,-2,-2,-2,0,5,37,5,15,1,0,1,4,39,6,1,0,1,0,2,41,6,1,1,0,2,-2
5191546,3754078,-2,1901,-2,21,full_end,-2,-1,-1,-1,-1,-2,2,94,26,-2,-2,-2,-2,-2,-2,0,5,37,5,15,1,0,1,4,39,6,1,0,1,0,2,41,6,1,1,0,3,-2
5191547,3754078,-2,1901,-2,21,full_end,-2,-2,-1,-1,-1,-2,2,94,26,-2,-2,-2,-2,-2,-2,0,5,37,5,15,1,0,1,4,39,6,1,0,1,0,2,41,6,1,1,0,4,0
5191548,3754078,-2,1901,-2,21,full_end,-2,-2,0,-1,-1,-2,2,94,26,-2,-2,-2,-2,-2,-2,0,5,37,5,15,1,0,1,4,39,6,1,0,1,0,2,41,6,1,1,0,5,-2


In [54]:
masked_df.to_csv('data/masked_df.csv')